In [ ]:
import math
import os.path as op
from glob import glob
from typing import List, Sequence

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch


In [ ]:
DATA_DIR = "" # Write your data path here
WORKING_DIR = "" # Write your working path here
FD_THRESH = 0.5
ALLOW_NA_FD = True

DSET_IMG_MAP = {
    "piop1": op.join(
        DATA_DIR,
        "aomic/halfpipe/piop1/derivatives/halfpipe/{subject}/func/task-workingmemory/{subject}_task-workingmemory_feature-wmVar1_taskcontrast-wmLoadVsControl_stat-z_statmap.nii.gz",
    ),
    "piop2": op.join(
        DATA_DIR,
        "aomic/halfpipe/piop2/derivatives/halfpipe/{subject}/func/task-workingmemory/{subject}_task-workingmemory_feature-wmVar1_taskcontrast-wmLoadVsControl_stat-z_statmap.nii.gz",
    ),
    "integramoods": op.join(
        DATA_DIR,
        "integramoods/halfpipe/*/*/derivatives/halfpipe/{subject}/func/task-nback/{subject}_task-nback_feature-nbackVar1_taskcontrast-wmLoadVsControl_stat-z_statmap.nii.gz",
    ),
    "hcpya": op.join(
        DATA_DIR,
        "hcp/halfpipe/young_adult/derivatives/halfpipe/{subject}/ses/func/task-WM/{subject}_task-WM_feature-WMVar1_taskcontrast-wmLoadVsControl_stat-z_statmap.nii.gz",
    ),
    "qtim": op.join(
        DATA_DIR,
        "queensland/qtim/halfpipe/derivatives/halfpipe/{subject}/ses-01/func/task-nback/{subject}_ses-01_task-nback_feature-wm1_taskcontrast-wmLoadVsControl_stat-z_statmap.nii.gz",
    ),
    "ds003849": op.join(
        DATA_DIR,
        "openneuro/ds003849/halfpipe/derivatives/halfpipe/{subject}/ses-01/func/task-nback/{subject}_ses-01_task-nback_run-01_feature-wmVar1_taskcontrast-wmLoadVsControl_stat-z_statmap.nii.gz",
    ),
    "wahn": op.join(
        DATA_DIR,
        "wahn/halfpipe/nback/derivatives/task_fmri-tasknm/{subject}_task-nback_feature-aroma_taskcontrast-wm_load_vs_control_stat-z_statmap.nii.gz",
    ),
    "pnc": op.join(
        DATA_DIR,
        "pnc/halfpipe/derivatives/halfpipe/{subject}/func/task-frac2back/{subject}_task-frac2back_feature-nbackVar1_taskcontrast-wmLoadVsControl_stat-z_statmap.nii.gz",
    ),
    "chcp": op.join(
        DATA_DIR,
        "hcp/raw/chinese/preproc/{subject}/MNINonLinear/Results/tfMRI_Nback/tfMRI_Nback_hp200_s2_level2.feat/StandardVolumeStats/cope7.feat/zstat1_s6.nii.gz",
    ),
    "abcd": op.join(
        DATA_DIR,
        "abcd/halfpipe/*/derivatives/halfpipe-fixed-effects/{subject}/ses-2YearFollowUpYArm1/func/task-nback/{subject}_ses-2YearFollowUpYArm1_task-nback_feature-nback1_taskcontrast-wmLoadVsControl_stat-z_statmap.nii.gz",
    ),
    "abcd-2025": op.join(
        DATA_DIR,
        "abcd/halfpipe-2025/derivatives/halfpipe-fixed-effects/{subject}/ses-{fu}/func/task-nback/{subject}_ses-{fu}_task-nback_feature-nback1_taskcontrast-wmLoadVsControl_stat-z_statmap.nii.gz",
    ),
}


def filter_task(df, task_name, add_na=False, fd_thresh=FD_THRESH):
    """
    Filter rows for a given task with good rating and FD threshold.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    task_name : str
        Task name to filter on.
    add_na : bool, optional
        If True, allow rows with NA `fd_mean` in addition to rows below threshold.

    Returns
    -------
    pandas.DataFrame
        Filtered dataframe.
    """
    mask = (df["task"] == task_name) & (df["rating"] == "good")
    if add_na:
        mask &= (df["fd_mean"] < fd_thresh) | (df["fd_mean"].isna())
    else:
        mask &= df["fd_mean"] < fd_thresh
    return df.loc[mask]


def aggregate_qc(
    qc_df: pd.DataFrame,
    by: Sequence[str],
) -> pd.DataFrame:
    """
    Aggregate QC metrics by group.

    Parameters
    ----------
    qc_df : pandas.DataFrame
        Input QC dataframe with columns `rating_col` and `fd_col`.

    Returns
    -------
    pandas.DataFrame
        Aggregated dataframe with:
        - `fd_mean`: mean of `fd_col`
        - `rating`: "good" if all ratings in group are "good" (case/space-insensitive), else "bad"
    """

    def _agg_rating(s: pd.Series) -> str:
        s = s.dropna().astype(str).str.strip().str.lower()
        return "good" if (len(s) > 0 and (s == "good").all()) else "bad"

    return (
        qc_df.groupby(list(by), dropna=False)
        .agg(
            fd_mean=("fd_mean", "mean"),
            rating=("rating", _agg_rating),
        )
        .reset_index()
    )


def pad4_or_keep(s):
    s = str(s)
    return f"sub-{int(s):04d}" if s.isdigit() else s


def sub_prefix(s):
    return f"sub-{s}"


def _read_table(path: str) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t" if path.endswith(".tsv") else ",")


def _safe_str(v):
    return "" if pd.isna(v) else str(v)


def _build_img_paths(df: pd.DataFrame, template: str) -> list:
    out = []
    for _, row in df.iterrows():
        subj = _safe_str(row.get("subject", ""))
        fu = _safe_str(row.get("fu", ""))
        try:
            pattern = template.format(subject=subj, fu=fu)
            m = glob(pattern)
            out.append(m[0] if m else None)
        except Exception:
            out.append(None)
    return out


def make_dataset(
    dset: str,
    beh_path: str,
    qc_path: str,
    metadata_template: pd.DataFrame,
    *,
    merge_on=("subject",),
    merge_how="left",
    qc_query="task == 'wm'",
    aggregate_by=None,
    drop_cols=(),
    subject_fmt=lambda s: s,
    img_template=None,
    dedup_subset=None,
    beh_transform=None,
    post=None,
    apply_fmt_to="qc",  # 'qc' | 'beh' | 'both' | None
) -> pd.DataFrame:
    """
    Build dataset metadata end-to-end.

    Parameters
    ----------
    dset : str
    beh_path : str
    qc_path : str
    metadata_template : pandas.DataFrame
    merge_on : tuple of str
    merge_how : str
    qc_query : str
    aggregate_by : tuple of str or None
    drop_cols : tuple of str
    subject_fmt : callable
        Function to normalize `subject`, applied pre-merge on side(s) per `apply_fmt_to`.
    img_template : str
    dedup_subset : tuple of str or None
    beh_transform : callable or None
    post : callable or None
    apply_fmt_to : str or None
        Which side to format pre-merge: 'qc', 'beh', 'both', or None.

    Returns
    -------
    pandas.DataFrame
    """
    beh_df = _read_table(op.join(WORKING_DIR, beh_path))
    if beh_transform is not None:
        beh_df = beh_transform(beh_df)

    qc_df = _read_table(op.join(WORKING_DIR, qc_path))
    if qc_query:
        qc_df = qc_df.query(qc_query)
    if aggregate_by:
        qc_df = aggregate_qc(qc_df, by=list(aggregate_by))

    # ensure dtype alignment and pre-merge subject formatting
    if "subject" in beh_df.columns:
        beh_df["subject"] = beh_df["subject"].astype(str)
    if "subject" in qc_df.columns:
        qc_df["subject"] = qc_df["subject"].astype(str)

    if (
        subject_fmt is not None
        and apply_fmt_to in ("qc", "both")
        and "subject" in qc_df.columns
    ):
        qc_df["subject"] = qc_df["subject"].map(subject_fmt)
    if (
        subject_fmt is not None
        and apply_fmt_to in ("beh", "both")
        and "subject" in beh_df.columns
    ):
        beh_df["subject"] = beh_df["subject"].map(subject_fmt)

    df = beh_df.merge(qc_df, on=list(merge_on), how=merge_how)

    df = filter_task(df, "wm", add_na=ALLOW_NA_FD, fd_thresh=FD_THRESH)
    if drop_cols:
        df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    if dedup_subset:
        df = df.drop_duplicates(subset=list(dedup_subset), keep="first")
        for c in dedup_subset:
            if c in df.columns:
                df = df.drop(columns=c)

    df = df.merge(metadata_template, on="site", how="left")

    if img_template is None:
        img_template = DSET_IMG_MAP[dset]
    df["fpath"] = _build_img_paths(df, img_template)
    df = df.dropna(subset=["fpath"])

    if post is not None:
        df = post(df)
    return df

In [ ]:
METADATA_DICT = {}

metadata_template = pd.read_csv(
    op.join(WORKING_DIR, "data", "metadata_template_wm.csv")
)
metadata_template

In [ ]:
# PIOP1
dset = "piop1"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path=f"dsets/{dset}_data.csv",
    qc_path=f"dsets/{dset}_qc_fd.csv",
    metadata_template=metadata_template,
    qc_query="task == 'wm'",
    drop_cols=("nback", "task"),
    subject_fmt=pad4_or_keep,
    img_template=DSET_IMG_MAP[dset],
)
METADATA_DICT[dset]


In [ ]:
# PIOP2
dset = "piop2"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path=f"dsets/{dset}_data.csv",
    qc_path=f"dsets/{dset}_qc_fd.csv",
    metadata_template=metadata_template,
    qc_query="task == 'wm'",
    drop_cols=("nback", "task"),
    subject_fmt=pad4_or_keep,
    img_template=DSET_IMG_MAP[dset],
)
METADATA_DICT[dset]


In [ ]:
# integramoods
dset = "integramoods"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path=f"dsets/{dset}_data.csv",
    qc_path=f"dsets/{dset}_qc_fd.csv",
    metadata_template=metadata_template,
    qc_query="task == 'wm'",
    drop_cols=("nback", "task", "mid", "group"),
    subject_fmt=sub_prefix,
    img_template=DSET_IMG_MAP[dset],
)
METADATA_DICT[dset]


In [ ]:
# hcp-ya
dset = "hcpya"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path=f"dsets/{dset}_data.csv",
    qc_path=f"dsets/{dset}_qc_fd.csv",
    metadata_template=metadata_template,
    qc_query="task == 'wm'",
    aggregate_by=("subject", "task"),
    drop_cols=("nback", "mid"),
    subject_fmt=sub_prefix,
    img_template=DSET_IMG_MAP[dset],
)
METADATA_DICT[dset]

In [ ]:
# qtim
dset = "qtim"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path=f"dsets/{dset}_data.csv",
    qc_path=f"dsets/{dset}_qc_fd.csv",
    metadata_template=metadata_template,
    qc_query="task == 'wm' and ses == 1",
    drop_cols=("nback", "ses", "task"),
    subject_fmt=pad4_or_keep,
    dedup_subset=("family_id",),
    img_template=DSET_IMG_MAP[dset],
)
METADATA_DICT[dset]

In [ ]:
# ds003849
dset = "ds003849"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path=f"dsets/{dset}_data.csv",
    qc_path=f"dsets/{dset}_qc_fd.csv",
    metadata_template=metadata_template,
    qc_query="task == 'wm'",
    drop_cols=("nback", "rating", "task"),
    subject_fmt=sub_prefix,
    apply_fmt_to="both",
    beh_transform=lambda df: df.assign(
        subject=df["subject"].astype(str) + "A",
        sex=df["sex"].map({0: "F", 1: "M"}).astype(str),
    ),
    img_template=DSET_IMG_MAP[dset],
)
METADATA_DICT[dset]

In [ ]:
# wahn
dset = "wahn"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path=f"dsets/{dset}_data.csv",
    qc_path=f"dsets/{dset}_qc_fd.csv",
    metadata_template=metadata_template,
    qc_query="task == 'wm'",
    drop_cols=("nback", "task"),
    subject_fmt=sub_prefix,
    img_template=DSET_IMG_MAP[dset],
)
METADATA_DICT[dset]

In [ ]:
# pnc
dset = "pnc"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path=f"dsets/{dset}_data.csv",
    qc_path=f"dsets/{dset}_qc_fd.csv",
    metadata_template=metadata_template,
    qc_query="task == 'wm'",
    drop_cols=("nback", "task"),
    subject_fmt=sub_prefix,
    img_template=DSET_IMG_MAP[dset],
)
METADATA_DICT[dset]

In [ ]:
# chcp
dset = "chcp"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path=f"dsets/{dset}_data.csv",
    qc_path=f"dsets/{dset}_qc.csv",
    metadata_template=metadata_template,
    qc_query="task == 'wm'",
    aggregate_by=("subject", "task"),
    drop_cols=("nback", "mid", "task"),
    subject_fmt=str,
    img_template=DSET_IMG_MAP[dset],
)
METADATA_DICT[dset]


In [ ]:
# abcd-2025
def _abcd_transform(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop(
        columns=[c for c in ["ehi_y_ss_scoreb"] if c in df.columns], errors="ignore"
    )
    df = df.rename(columns={"demo_sex_v2": "sex", "interview_age": "age"})
    if "age" in df.columns:
        df["age"] = df["age"] / 12.0
    return df


dset = "abcd"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path="dsets/abcd-all_data.csv",
    qc_path="dsets/abcd-final_qc_fd.csv",
    metadata_template=metadata_template,
    merge_on=("subject", "fu"),
    merge_how="right",
    qc_query="task == 'wm' and preproc_year == 2025",
    drop_cols=("preproc_year", "scanner", "task"),
    subject_fmt=sub_prefix,
    apply_fmt_to="both",  # key tweak
    img_template=DSET_IMG_MAP["abcd-2025"],
    beh_transform=_abcd_transform,
    post=lambda df: df.drop(columns=[c for c in ["fu"] if c in df.columns]),
)
METADATA_DICT[dset]

In [ ]:
# abcd-2023
dset = "abcd_2023"
METADATA_DICT[dset] = make_dataset(
    dset,
    beh_path="dsets/abcd-all_data.csv",
    qc_path="dsets/abcd-final_qc_fd.csv",
    metadata_template=metadata_template,
    merge_on=("subject", "fu"),
    merge_how="right",
    qc_query="task == 'wm' and preproc_year == 2023",
    drop_cols=("fu", "preproc_year", "scanner", "task"),
    subject_fmt=sub_prefix,
    apply_fmt_to="both",
    img_template=DSET_IMG_MAP["abcd"],
    beh_transform=_abcd_transform,
)
METADATA_DICT[dset]

In [ ]:
# Concatenate all DataFrames from METADATA_DICT into a single DataFrame
all_metadata_df = pd.concat(METADATA_DICT.values(), ignore_index=True).drop(
    columns=["task"]
)
all_metadata_df = all_metadata_df[
    all_metadata_df["site"].notna() & (all_metadata_df["site"] != "")
]
all_metadata_df = all_metadata_df[all_metadata_df["sex"] != "3"]
all_metadata_df.to_csv(
    op.join(WORKING_DIR, "data", "metadata_wm-loadVsControl.csv"), index=False
)
all_metadata_df


In [ ]:
def hist_by(df: pd.DataFrame, x: str, hue: str, title: str, xlabel: str) -> None:
    """
    Histogram with stacked hue.

    Parameters
    ----------
    df : pandas.DataFrame
    x : str
        Column to plot on x-axis.
    hue : str
        Column for hue.
    title : str
    xlabel : str
    """
    _, ax = plt.subplots(figsize=(15, 10), dpi=300)

    # determine categories in the same order they appear
    sites = df[hue].dropna().astype(str).unique().tolist()

    # use a qualitative palette designed for many distinct categories
    # 'husl' spa ces hues evenly and remains fairly distinguishable up to many colors
    palette = sns.color_palette("husl", n_colors=len(sites))

    sns.histplot(
        data=df,
        x=x,
        hue=hue,
        bins=60,
        ax=ax,
        stat="count",
        multiple="stack",
        common_norm=False,
        alpha=0.85,
        linewidth=1.0,
        edgecolor="black",
        palette=palette,
    )

    handles, labels = ax.get_legend_handles_labels()

    # If seaborn didn't produce a usable legend, build one from our palette
    if not labels or len(labels) != len(sites):
        handles = [
            Patch(facecolor=palette[i], edgecolor="black", alpha=0.85)
            for i in range(len(sites))
        ]
        labels = sites

    ax.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.2),
        ncol=5,
        frameon=False,
    )
    plt.subplots_adjust(bottom=0.18)
    ax.set_xlabel(xlabel, fontsize=16, labelpad=12)
    ax.set_ylabel("Count", fontsize=16, labelpad=12)
    ax.set_title(title, fontsize=20, pad=20)
    ax.tick_params(axis="both", which="major", labelsize=13)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=25))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6))
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.show()


def plot_var_grid(df: pd.DataFrame, vars_to_plot: List[str], ncols: int = 3) -> None:
    """
    Small-multiples bar counts for many variables.

    Parameters
    ----------
    df : pandas.DataFrame
    vars_to_plot : list of str
    ncols : int
    """
    n_vars = len(vars_to_plot)
    nrows = math.ceil(n_vars / ncols)
    _, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 5 * nrows), dpi=150)
    axes = axes.flatten()

    for i, var in enumerate(vars_to_plot):
        ax = axes[i]
        if var not in df.columns:
            ax.set_title(f"{var} (missing)", fontsize=16)
            ax.axis("off")
            continue

        s = df[var].dropna()
        if pd.api.types.is_numeric_dtype(s) and s.nunique() > 20:
            counts = pd.cut(s, bins=20).value_counts(sort=False)
            x = counts.index.astype(str)
        else:
            counts = s.astype(str).value_counts()
            if len(counts) > 30:
                counts = counts.head(30)
            x = counts.index

        n_bars = len(counts)
        ax.bar(x, counts.values, color=sns.color_palette("tab20", n_colors=n_bars))
        ax.set_title(var, fontsize=16)
        ax.set_ylabel("Count", fontsize=14)
        ax.tick_params(axis="x", rotation=45, labelsize=10)
        ax.tick_params(axis="y", labelsize=12)
        ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
        # ensure top tick ≥ tallest bar
        # top = int(np.nanmax(counts.values)) if len(counts) else 0
        # yloc = plt.MaxNLocator(nbins=5, integer=True)
        # ticks = yloc.tick_values(0, top)
        # ax.set_ylim(0, ticks[-1])
        # ax.yaxis.set_major_locator(yloc)

    for ax in axes[n_vars:]:
        ax.axis("off")

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.06)
    plt.show()


all_metadata_df = pd.read_csv(op.join(WORKING_DIR, "data", "metadata_wm.csv"))

# Calls (replace your three plotting cells)
hist_by(
    all_metadata_df,
    x="age",
    hue="sex",
    title="Age Distribution by Sex",
    xlabel="Age (years)",
)
hist_by(
    all_metadata_df,
    x="age",
    hue="site",
    title="Age Distribution by Site",
    xlabel="Age (years)",
)
hist_by(
    all_metadata_df,
    x="fd_mean",
    hue="site",
    title="FD Distribution by Site",
    xlabel="Framewise Displacement (mm)",
)

plot_var_grid(
    all_metadata_df,
    vars_to_plot=[
        "sex",
        "scanner",
        "tr",
        "te",
        "mb_factor",
        "pa_factor",
        "phase_enc_dir_lr_rl",
        "phase_enc_dir_ap",
        "phase_enc_dir_pa",
        "fixed_effect",
        "volumes",
        "halfpipe",
        "task_length_s",
        "task_design",
        "target_length_ms",
        "target_stimuli",
        "n_trials",
    ],
    ncols=3,
)